<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/BindingDB/BindingDB_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import zipfile
import re
import os

# ============================================================
# INPUT / OUTPUT
# ============================================================

ZIP_PATH = "/content/BindingDB_All_202608_tsv.zip"

OUT_ALL = "/content/BindingDB_Kinase_MetabolicEnzyme_Kd.tsv.gz"
OUT_KINASE = "/content/BindingDB_Kinase_Kd.tsv.gz"
OUT_METABOLIC = "/content/BindingDB_MetabolicEnzyme_Kd.tsv.gz"


# ============================================================
# 1. FIND TSV INSIDE ZIP
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()
    print("Files inside ZIP:")
    print(files)

    tsv_files = [x for x in files if x.lower().endswith(".tsv")]

    if not tsv_files:
        raise ValueError("No TSV file found inside ZIP.")

    TSV_NAME = tsv_files[0]

print("\nUsing:", TSV_NAME)


# ============================================================
# 2. READ HEADER ONLY
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open(TSV_NAME) as f:
        header = pd.read_csv(
            f,
            sep="\t",
            nrows=0
        ).columns.tolist()

print("\nNumber of columns:", len(header))


# ============================================================
# 3. FIND IMPORTANT COLUMNS ROBUSTLY
# ============================================================

def find_col(possible_names, required=False):
    for candidate in possible_names:
        for actual in header:
            if actual.strip() == candidate.strip():
                return actual

    # fallback: normalized comparison
    def norm(x):
        return re.sub(r"\s+", " ", x.strip().lower())

    for candidate in possible_names:
        for actual in header:
            if norm(actual) == norm(candidate):
                return actual

    if required:
        raise ValueError(
            f"Could not find required column. Tried:\n{possible_names}"
        )

    return None


COL_KD = find_col(
    ["Kd (nM)"],
    required=True
)

COL_TARGET = find_col(
    ["Target Name"],
    required=True
)

COL_SEQ = find_col([
    "BindingDB Target Chain Sequence",
    "BindingDB Target Chain  Sequence",
    "BindingDB Target Chain Sequence 1",
    "BindingDB Target Chain  Sequence 1",
])

COL_UNIPROT_NAME = find_col([
    "UniProt (SwissProt) Recommended Name of Target Chain",
    "UniProt (SwissProt) Recommended Name of Target Chain 1",
])

COL_UNIPROT_ENTRY = find_col([
    "UniProt (SwissProt) Entry Name of Target Chain",
    "UniProt (SwissProt) Entry Name of Target Chain 1",
])

COL_UNIPROT_ID = find_col([
    "UniProt (SwissProt) Primary ID of Target Chain",
    "UniProt (SwissProt) Primary ID of Target Chain 1",
])

COL_SMILES = find_col(["Ligand SMILES"])
COL_INCHIKEY = find_col([
    "Ligand InChI Key",
    "Ligand InChI key",
])

COL_LIGAND = find_col([
    "BindingDB Ligand Name"
])

COL_RSID = find_col([
    "BindingDB Reactant_set_id"
])

COL_ORGANISM = find_col([
    "Target Source Organism According to Curator or DataSource"
])

COL_DOI = find_col(["Article DOI"])
COL_PMID = find_col(["PMID"])

COL_SOURCE = find_col([
    "Curation/DataSource"
])

COL_PDB = find_col([
    "PDB ID(s) for Ligand-Target Complex"
])


print("\nDetected columns:")
for name, col in {
    "Kd": COL_KD,
    "Target": COL_TARGET,
    "Sequence": COL_SEQ,
    "UniProt Name": COL_UNIPROT_NAME,
    "UniProt Entry": COL_UNIPROT_ENTRY,
    "UniProt ID": COL_UNIPROT_ID,
    "SMILES": COL_SMILES,
    "InChIKey": COL_INCHIKEY,
}.items():
    print(f"{name:15s}: {col}")

Files inside ZIP:
['BindingDB_All.tsv']

Using: BindingDB_All.tsv

Number of columns: 640

Detected columns:
Kd             : Kd (nM)
Target         : Target Name
Sequence       : BindingDB Target Chain Sequence 1
UniProt Name   : UniProt (SwissProt) Recommended Name of Target Chain 1
UniProt Entry  : UniProt (SwissProt) Entry Name of Target Chain 1
UniProt ID     : UniProt (SwissProt) Primary ID of Target Chain 1
SMILES         : Ligand SMILES
InChIKey       : Ligand InChI Key


In [ ]:
# ============================================================
# 4. TARGET CLASSIFICATION RULES
# ============================================================

# Kinase: intentionally fairly strict.
KINASE_PATTERN = re.compile(
    r"""
    \bkinase\b
    |\bkinases\b
    |\btyrosine[- ]protein\s+kinase\b
    |\bserine/threonine[- ]protein\s+kinase\b
    |\bprotein[- ]tyrosine\s+kinase\b
    |\bphosphoinositide\s+3[- ]kinase\b
    |\bPI3[- ]kinase\b
    """,
    re.I | re.X
)


# Metabolic enzyme patterns.
# This is deliberately broader but excludes obvious proteases below.
METABOLIC_PATTERN = re.compile(
    r"""
    dehydrogenase
    |reductase
    |oxidase
    |oxygenase
    |monooxygenase
    |dioxygenase
    |synthase
    |synthetase
    |transferase
    |methyltransferase
    |acetyltransferase
    |glycosyltransferase
    |aminotransferase
    |transaminase
    |carboxylase
    |decarboxylase
    |deaminase
    |isomerase
    |epimerase
    |racemase
    |aldolase
    |enolase
    |hydrolase
    |phosphatase
    |phosphodiesterase
    |phospholipase
    |esterase
    |lipase
    |glycosidase
    |glucosidase
    |galactosidase
    |mannosidase
    |sialidase
    |anhydrase
    |ligase
    |lyase
    |cytochrome\s+P450
    |aromatase
    |cyclooxygenase
    |lipoxygenase
    |ATPase
    """,
    re.I | re.X
)


# Keep proteases out of the metabolic-enzyme group.
PROTEASE_EXCLUDE = re.compile(
    r"""
    protease
    |proteinase
    |peptidase
    |caspase
    |cathepsin
    |trypsin
    |chymotrypsin
    |thrombin
    |elastase
    |matrix\s+metalloproteinase
    |\bMMP[- ]?\d*
    """,
    re.I | re.X
)


def classify_target(text):
    text = str(text)

    if KINASE_PATTERN.search(text):
        return "Kinase"

    if (
        METABOLIC_PATTERN.search(text)
        and not PROTEASE_EXCLUDE.search(text)
    ):
        return "Metabolic enzyme"

    return None

In [ ]:
# ============================================================
# 5. COLUMNS TO RETAIN
# ============================================================

keep_candidates = [
    COL_RSID,
    COL_LIGAND,
    COL_SMILES,
    COL_INCHIKEY,

    COL_TARGET,
    COL_ORGANISM,

    COL_SEQ,

    COL_UNIPROT_NAME,
    COL_UNIPROT_ENTRY,
    COL_UNIPROT_ID,

    COL_KD,

    COL_PDB,
    COL_SOURCE,
    COL_DOI,
    COL_PMID,
]

KEEP_COLS = [
    c for c in keep_candidates
    if c is not None
]

# Remove duplicates while preserving order
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))

print("Columns that will be retained:")
for c in KEEP_COLS:
    print("  ", c)

Columns that will be retained:
   BindingDB Reactant_set_id
   BindingDB Ligand Name
   Ligand SMILES
   Ligand InChI Key
   Target Name
   Target Source Organism According to Curator or DataSource
   BindingDB Target Chain Sequence 1
   UniProt (SwissProt) Recommended Name of Target Chain 1
   UniProt (SwissProt) Entry Name of Target Chain 1
   UniProt (SwissProt) Primary ID of Target Chain 1
   Kd (nM)
   PDB ID(s) for Ligand-Target Complex
   Curation/DataSource
   Article DOI
   PMID


In [ ]:
# ============================================================
# 6. PROCESS BINDINGDB IN CHUNKS
# ============================================================

chunksize = 100_000

first_all = True
first_kinase = True
first_metabolic = True

n_total = 0
n_kd = 0
n_kinase = 0
n_metabolic = 0


with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open(TSV_NAME) as f:

        reader = pd.read_csv(
            f,
            sep="\t",
            usecols=KEEP_COLS,
            chunksize=chunksize,
            dtype=str,
            low_memory=False,
            on_bad_lines="skip"
        )

        for i, chunk in enumerate(reader, start=1):

            n_total += len(chunk)

            # ------------------------------------------------
            # KEEP ONLY ROWS HAVING Kd
            # ------------------------------------------------

            kd_text = (
                chunk[COL_KD]
                .astype("string")
                .str.strip()
            )

            mask_kd = (
                kd_text.notna()
                & (kd_text != "")
                & (kd_text.str.lower() != "nan")
            )

            chunk = chunk.loc[mask_kd].copy()

            n_kd += len(chunk)

            if len(chunk) == 0:
                continue


            # ------------------------------------------------
            # BUILD TARGET CLASSIFICATION TEXT
            # only from fields INSIDE BindingDB_All.tsv
            # ------------------------------------------------

            target_parts = []

            for c in [
                COL_TARGET,
                COL_UNIPROT_NAME,
                COL_UNIPROT_ENTRY
            ]:
                if c is not None and c in chunk.columns:
                    target_parts.append(
                        chunk[c].fillna("").astype(str)
                    )

            target_text = target_parts[0]

            for s in target_parts[1:]:
                target_text = target_text + " | " + s


            # ------------------------------------------------
            # CLASSIFY
            # ------------------------------------------------

            chunk["Target_Class"] = target_text.map(classify_target)

            selected = chunk[
                chunk["Target_Class"].notna()
            ].copy()

            if len(selected) == 0:
                continue


            # ------------------------------------------------
            # PARSE Kd, while preserving original BindingDB value
            #
            # Examples:
            #   15
            #   <10
            #   >10000
            # ------------------------------------------------

            selected["Kd_original"] = selected[COL_KD]

            selected["Kd_qualifier"] = (
                selected[COL_KD]
                .astype(str)
                .str.extract(r"^\s*([<>~=]+)", expand=False)
                .fillna("=")
            )

            selected["Kd_nM_numeric"] = pd.to_numeric(
                selected[COL_KD]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.extract(
                    r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
                    expand=False
                ),
                errors="coerce"
            )


            # ------------------------------------------------
            # SPLIT CLASSES
            # ------------------------------------------------

            kinase = selected[
                selected["Target_Class"] == "Kinase"
            ].copy()

            metabolic = selected[
                selected["Target_Class"] == "Metabolic enzyme"
            ].copy()

            n_kinase += len(kinase)
            n_metabolic += len(metabolic)


            # ------------------------------------------------
            # WRITE
            # ------------------------------------------------

            selected.to_csv(
                OUT_ALL,
                sep="\t",
                index=False,
                compression="gzip",
                mode="wt" if first_all else "at",
                header=first_all
            )

            first_all = False


            if len(kinase):
                kinase.to_csv(
                    OUT_KINASE,
                    sep="\t",
                    index=False,
                    compression="gzip",
                    mode="wt" if first_kinase else "at",
                    header=first_kinase
                )
                first_kinase = False


            if len(metabolic):
                metabolic.to_csv(
                    OUT_METABOLIC,
                    sep="\t",
                    index=False,
                    compression="gzip",
                    mode="wt" if first_metabolic else "at",
                    header=first_metabolic
                )
                first_metabolic = False


            print(
                f"Chunk {i:3d} | "
                f"scanned {n_total:,} | "
                f"Kd {n_kd:,} | "
                f"kinase {n_kinase:,} | "
                f"metabolic {n_metabolic:,}"
            )


print("\nDONE")
print("Total BindingDB rows:", f"{n_total:,}")
print("Rows with Kd:", f"{n_kd:,}")
print("Kinase Kd rows:", f"{n_kinase:,}")
print("Metabolic enzyme Kd rows:", f"{n_metabolic:,}")

print("\nFiles:")
print(OUT_ALL)
print(OUT_KINASE)
print(OUT_METABOLIC)

Chunk   1 | scanned 100,000 | Kd 3,714 | kinase 2,506 | metabolic 131
Chunk   2 | scanned 200,000 | Kd 5,018 | kinase 2,806 | metabolic 453
Chunk   3 | scanned 300,000 | Kd 6,793 | kinase 3,119 | metabolic 675
Chunk   4 | scanned 400,000 | Kd 8,422 | kinase 3,622 | metabolic 847
Chunk   5 | scanned 500,000 | Kd 8,817 | kinase 3,641 | metabolic 1,089
Chunk   6 | scanned 600,000 | Kd 10,102 | kinase 4,306 | metabolic 1,414
Chunk   7 | scanned 700,000 | Kd 12,282 | kinase 5,767 | metabolic 1,422
Chunk   8 | scanned 800,000 | Kd 12,721 | kinase 6,038 | metabolic 1,462
Chunk   9 | scanned 900,000 | Kd 15,936 | kinase 8,916 | metabolic 1,462
Chunk  10 | scanned 1,000,000 | Kd 17,460 | kinase 9,896 | metabolic 1,562
Chunk  11 | scanned 1,100,000 | Kd 21,248 | kinase 11,659 | metabolic 1,578
Chunk  12 | scanned 1,200,000 | Kd 25,403 | kinase 13,828 | metabolic 1,578
Chunk  13 | scanned 1,300,000 | Kd 27,915 | kinase 14,478 | metabolic 1,881
Chunk  14 | scanned 1,400,000 | Kd 32,260 | kinase 15

In [ ]:
import pandas as pd
import zipfile
import re
import os

# ============================================================
# INPUT / OUTPUT
# ============================================================

ZIP_PATH = "/content/BindingDB_All_202608_tsv.zip"

OUT_KINASE = "/content/BindingDB_Kinase_Kd.csv.gz"
OUT_METABOLIC = "/content/BindingDB_MetabolicEnzyme_Kd.csv.gz"


# ============================================================
# 1. FIND TSV INSIDE ZIP
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    tsv_files = [
        x for x in z.namelist()
        if x.lower().endswith(".tsv")
    ]

    if not tsv_files:
        raise ValueError("No TSV file found inside ZIP.")

    TSV_NAME = tsv_files[0]

print("Using:", TSV_NAME)


# ============================================================
# 2. READ HEADER
# ============================================================

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open(TSV_NAME) as f:
        columns = pd.read_csv(
            f,
            sep="\t",
            nrows=0
        ).columns.tolist()

print("Original BindingDB columns:", len(columns))


# ============================================================
# 3. FIND REQUIRED COLUMNS
# ============================================================

def find_column(exact_name):
    for col in columns:
        if col.strip().lower() == exact_name.strip().lower():
            return col
    raise ValueError(f"Cannot find column: {exact_name}")


COL_KD = find_column("Kd (nM)")
COL_TARGET = find_column("Target Name")

print("Kd column:", COL_KD)
print("Target column:", COL_TARGET)


# ============================================================
# 4. FIND OPTIONAL TARGET-DESCRIPTION COLUMNS
#
# These are ONLY used for classification.
# Nothing is joined from another database.
# ============================================================

def find_optional(text):
    matches = [
        c for c in columns
        if text.lower() in c.lower()
    ]
    return matches


target_text_columns = [COL_TARGET]

# Include protein names already present in BindingDB_All
for search_text in [
    "UniProt (SwissProt) Recommended Name",
    "UniProt (TrEMBL) Submitted Name"
]:
    target_text_columns.extend(find_optional(search_text))

# remove duplicate names
target_text_columns = list(dict.fromkeys(target_text_columns))

print("\nColumns used for target classification:")
for col in target_text_columns:
    print(" -", col)


# ============================================================
# 5. CLASSIFICATION RULES
# ============================================================

KINASE_PATTERN = re.compile(
    r"""
    \bkinase\b
    |\bkinases\b
    |\bprotein[- ]kinase\b
    |\bprotein[- ]tyrosine[- ]kinase\b
    |\btyrosine[- ]protein[- ]kinase\b
    |\bserine/threonine[- ]protein[- ]kinase\b
    |\bphosphatidylinositol[- ]?\d*[- ]kinase\b
    |\bphosphoinositide[- ]?\d*[- ]kinase\b
    """,
    re.I | re.X
)


METABOLIC_PATTERN = re.compile(
    r"""
    dehydrogenase
    |reductase
    |oxidase
    |oxygenase
    |monooxygenase
    |dioxygenase
    |synthase
    |synthetase
    |transferase
    |methyltransferase
    |acetyltransferase
    |glycosyltransferase
    |aminotransferase
    |transaminase
    |carboxylase
    |decarboxylase
    |deaminase
    |isomerase
    |epimerase
    |racemase
    |aldolase
    |enolase
    |hydrolase
    |phosphatase
    |phosphodiesterase
    |phospholipase
    |esterase
    |lipase
    |glycosidase
    |glucosidase
    |galactosidase
    |mannosidase
    |sialidase
    |anhydrase
    |ligase
    |lyase
    |cytochrome\s+P450
    |aromatase
    |cyclooxygenase
    |lipoxygenase
    |ATPase
    """,
    re.I | re.X
)


# Exclude proteases because you want a separate metabolic-enzyme class
PROTEASE_PATTERN = re.compile(
    r"""
    protease
    |proteinase
    |peptidase
    |caspase
    |cathepsin
    |trypsin
    |chymotrypsin
    |thrombin
    |elastase
    |metalloproteinase
    """,
    re.I | re.X
)


# ============================================================
# 6. PROCESS ENTIRE BINDINGDB FILE
# ============================================================

CHUNK_SIZE = 100_000

first_kinase = True
first_metabolic = True

n_total = 0
n_kd = 0
n_kinase = 0
n_metabolic = 0


with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open(TSV_NAME) as f:

        reader = pd.read_csv(
            f,
            sep="\t",

            # IMPORTANT:
            # no usecols => KEEP ALL BindingDB FIELDS
            chunksize=CHUNK_SIZE,

            dtype=str,
            low_memory=False,
            on_bad_lines="skip"
        )

        for chunk_num, chunk in enumerate(reader, 1):

            n_total += len(chunk)

            # =================================================
            # KEEP ONLY ROWS WITH Kd
            # =================================================

            kd = chunk[COL_KD].astype("string").str.strip()

            kd_mask = (
                kd.notna()
                & (kd != "")
                & (kd.str.lower() != "nan")
            )

            chunk = chunk.loc[kd_mask].copy()

            n_kd += len(chunk)

            if chunk.empty:
                continue


            # =================================================
            # BUILD TARGET DESCRIPTION
            # =================================================

            target_text = pd.Series(
                "",
                index=chunk.index,
                dtype="string"
            )

            for col in target_text_columns:
                if col in chunk.columns:
                    target_text = (
                        target_text
                        + " "
                        + chunk[col].fillna("").astype(str)
                    )


            # =================================================
            # IDENTIFY KINASES
            # =================================================

            kinase_mask = target_text.str.contains(
                KINASE_PATTERN,
                na=False
            )


            # =================================================
            # IDENTIFY METABOLIC ENZYMES
            # =================================================

            metabolic_mask = (
                target_text.str.contains(
                    METABOLIC_PATTERN,
                    na=False
                )
                &
                ~target_text.str.contains(
                    PROTEASE_PATTERN,
                    na=False
                )
                &
                ~kinase_mask
            )


            # =================================================
            # CREATE TWO TABLES
            # =================================================

            kinase = chunk.loc[kinase_mask].copy()
            metabolic = chunk.loc[metabolic_mask].copy()


            # =================================================
            # ADD THE 3 REQUESTED COLUMNS
            # =================================================

            for df, target_class in [
                (kinase, "Kinase"),
                (metabolic, "Metabolic enzyme")
            ]:

                if df.empty:
                    continue

                df["Target_Class"] = target_class

                # Capture <, >, <=, >=, ~ etc.
                df["Kd_qualifier"] = (
                    df[COL_KD]
                    .astype(str)
                    .str.extract(
                        r"^\s*([<>]=?|~|=)",
                        expand=False
                    )
                    .fillna("=")
                )

                # Extract numeric component
                df["Kd_nM_numeric"] = pd.to_numeric(
                    df[COL_KD]
                    .astype(str)
                    .str.replace(",", "", regex=False)
                    .str.extract(
                        r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
                        expand=False
                    ),
                    errors="coerce"
                )


            # =================================================
            # WRITE KINASE
            # =================================================

            if not kinase.empty:

                kinase.to_csv(
                    OUT_KINASE,
                    index=False,
                    compression="gzip",
                    mode="wt" if first_kinase else "at",
                    header=first_kinase
                )

                first_kinase = False
                n_kinase += len(kinase)


            # =================================================
            # WRITE METABOLIC ENZYME
            # =================================================

            if not metabolic.empty:

                metabolic.to_csv(
                    OUT_METABOLIC,
                    index=False,
                    compression="gzip",
                    mode="wt" if first_metabolic else "at",
                    header=first_metabolic
                )

                first_metabolic = False
                n_metabolic += len(metabolic)


            print(
                f"Chunk {chunk_num:>3} | "
                f"scanned={n_total:,} | "
                f"Kd={n_kd:,} | "
                f"kinase={n_kinase:,} | "
                f"metabolic={n_metabolic:,}"
            )


# ============================================================
# 7. SUMMARY
# ============================================================

print("\n================ DONE ================")

print(f"Total BindingDB rows scanned : {n_total:,}")
print(f"Rows containing Kd          : {n_kd:,}")
print(f"Kinase Kd rows              : {n_kinase:,}")
print(f"Metabolic enzyme Kd rows    : {n_metabolic:,}")

print("\nOutput:")
print(OUT_KINASE)
print(OUT_METABOLIC)

print("\nFile sizes:")
for path in [OUT_KINASE, OUT_METABOLIC]:
    if os.path.exists(path):
        print(
            os.path.basename(path),
            f"{os.path.getsize(path)/1024**2:.1f} MB"
        )

Using: BindingDB_All.tsv
Original BindingDB columns: 640
Kd column: Kd (nM)
Target column: Target Name

Columns used for target classification:
 - Target Name
 - UniProt (SwissProt) Recommended Name of Target Chain 1
 - UniProt (SwissProt) Recommended Name of Target Chain 2
 - UniProt (SwissProt) Recommended Name of Target Chain 3
 - UniProt (SwissProt) Recommended Name of Target Chain 4
 - UniProt (SwissProt) Recommended Name of Target Chain 5
 - UniProt (SwissProt) Recommended Name of Target Chain 6
 - UniProt (SwissProt) Recommended Name of Target Chain 7
 - UniProt (SwissProt) Recommended Name of Target Chain 8
 - UniProt (SwissProt) Recommended Name of Target Chain 9
 - UniProt (SwissProt) Recommended Name of Target Chain 10
 - UniProt (SwissProt) Recommended Name of Target Chain 11
 - UniProt (SwissProt) Recommended Name of Target Chain 12
 - UniProt (SwissProt) Recommended Name of Target Chain 13
 - UniProt (SwissProt) Recommended Name of Target Chain 14
 - UniProt (SwissProt) R

In [ ]:
import pandas as pd

file_path = "/content/BindingDB_Kinase_Kd.csv.gz"

df = pd.read_csv(file_path, compression="gzip", low_memory=False)

source_cols = [
    c for c in df.columns
    if any(x in c.lower() for x in [
        "doi",
        "pmid",
        "patent",
        "source",
        "author",
        "institution",
        "ph",
        "temp"
    ])
]

print(source_cols)
import numpy as np

def make_source_id(row):
    doi = str(row.get("Article DOI", "")).strip()
    pmid = str(row.get("PMID", "")).strip()
    patent = str(row.get("Patent Number", "")).strip()

    if doi and doi.lower() != "nan":
        return "DOI:" + doi.lower()

    if pmid and pmid.lower() != "nan":
        return "PMID:" + pmid

    if patent and patent.lower() != "nan":
        return "PATENT:" + patent

    return np.nan


df["Source_ID"] = df.apply(make_source_id, axis=1)
source_counts = (
    df["Source_ID"]
    .value_counts(dropna=True)
    .reset_index()
)

source_counts.columns = ["Source_ID", "N_rows"]

display(source_counts.head(20))

['Target Source Organism According to Curator or DataSource', 'pH', 'Temp (C)', 'Curation/DataSource', 'Article DOI', 'BindingDB Entry DOI', 'PMID', 'Patent Number', 'Authors', 'Institution', 'IUPHAR_GRAC ID of Ligand']


,Source_ID,N_rows
0,DOI:10.1038/nbt.1990,26856
1,DOI:10.1038/nbt1358,9856
2,DOI:10.1182/blood-2009-05-222034,2170
3,PATENT:US10508120,1366
4,PATENT:US10577373,1366
5,PATENT:US11046698,1366
6,PATENT:US9730937,1155
7,PATENT:US12221453,1027
8,PATENT:US11414431,1027
9,PATENT:US20250074891,1001


In [ ]:
import pandas as pd
import numpy as np

file_path = "/content/BindingDB_Kinase_Kd.csv.gz"

df = pd.read_csv(
    file_path,
    compression="gzip",
    low_memory=False
)

# Check which source-ID columns actually exist
candidate_cols = [
    "Article DOI",
    "PMID",
    "Patent Number",
    "BindingDB Entry DOI",
    "PubChem AID"
]

source_cols = [c for c in candidate_cols if c in df.columns]

print("Using source columns:")
for c in source_cols:
    print(" -", c)

Using source columns:
 - Article DOI
 - PMID
 - Patent Number
 - BindingDB Entry DOI
 - PubChem AID


In [ ]:
def clean_id(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() in ["", "nan", "none"]:
        return ""

    return x.lower()


for col in source_cols:
    df[col + "_clean"] = df[col].map(clean_id)

In [ ]:
def build_combined_source_id(row):
    parts = []

    doi = row.get("Article DOI_clean", "")
    pmid = row.get("PMID_clean", "")
    patent = row.get("Patent Number_clean", "")
    entry_doi = row.get("BindingDB Entry DOI_clean", "")
    aid = row.get("PubChem AID_clean", "")

    if doi:
        parts.append("DOI=" + doi)

    if pmid:
        parts.append("PMID=" + pmid)

    if patent:
        parts.append("PATENT=" + patent)

    if entry_doi:
        parts.append("BDBDOI=" + entry_doi)

    if aid:
        parts.append("AID=" + aid)

    if not parts:
        return np.nan

    return " | ".join(parts)


df["Combined_Source_ID"] = df.apply(
    build_combined_source_id,
    axis=1
)

In [ ]:
source_summary = (
    df.groupby("Combined_Source_ID", dropna=True)
      .size()
      .reset_index(name="N_rows")
      .sort_values("N_rows", ascending=False)
)

display(source_summary.head(30))

,Combined_Source_ID,N_rows
677,DOI=10.1038/nbt.1990 | PMID=22037378.0 | BDBDOI=10.7270/q25d8s70,26856
679,DOI=10.1038/nbt1358 | PMID=18183025.0 | BDBDOI=10.7270/q2tt4rx2,9856
701,DOI=10.1182/blood-2009-05-222034 | PMID=19654408.0 | BDBDOI=10.7270/q2pn95v2,2170
715,PATENT=us10508120 | BDBDOI=10.7270/q29k4dmt | AID=aid1804058,1366
739,PATENT=us11046698 | BDBDOI=10.7270/q2xg9v85 | AID=aid1805721,1366
719,PATENT=us10577373 | BDBDOI=10.7270/q2cr5wr4 | AID=aid1804241,1366
831,PATENT=us9730937 | BDBDOI=10.7270/q2m32xvc | AID=aid1796958,1155
754,PATENT=us11414431 | BDBDOI=10.7270/q2rx9g90 | AID=aid1806738,1027
769,PATENT=us12221453 | BDBDOI=10.7270/q21g0rrq | AID=aid2061008,1027
788,PATENT=us20250074891 | BDBDOI=10.7270/q2sj1r50,1001


In [ ]:
import pandas as pd

file_path = "/content/BindingDB_Kinase_Kd.csv.gz"

df = pd.read_csv(
    file_path,
    compression="gzip",
    low_memory=False
)

doi = "10.1038/nbt.1990"

x = df[
    df["Article DOI"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq(doi)
].copy()

print("Rows:", len(x))

print("\nUnique authors:")
print(x["Authors"].nunique(dropna=True))

print("\nUnique ligands:")
print(x["BindingDB MonomerID"].nunique(dropna=True))

print("\nUnique target names:")
print(x["Target Name"].nunique(dropna=True))

Rows: 26856

Unique authors:
1

Unique ligands:
72

Unique target names:
330


In [ ]:
import pandas as pd
import numpy as np
import re
import os


# ============================================================
# INPUT / OUTPUT
# ============================================================

FILES = {
    "Kinase": {
        "input": "/content/BindingDB_Kinase_Kd.csv.gz",
        "output": "/content/BindingDB_Kinase_Kd_WT_MT_same_source.csv.gz",
    },

    "Metabolic enzyme": {
        "input": "/content/BindingDB_MetabolicEnzyme_Kd.csv.gz",
        "output": "/content/BindingDB_MetabolicEnzyme_Kd_WT_MT_same_source.csv.gz",
    }
}

CHUNK_SIZE = 100_000


# ============================================================
# GENERAL CLEANING
# ============================================================

def clean(x):
    if pd.isna(x):
        return ""

    x = str(x).strip()

    if x.lower() in ["", "nan", "none", "null"]:
        return ""

    return x


# ============================================================
# SOURCE ID NORMALIZATION
#
# We combine:
#   Article DOI
#   PMID
#   Patent Number
#   BindingDB Entry DOI
#   PubChem AID
#
# Authors are deliberately NOT used.
# ============================================================

SOURCE_COLUMNS = [
    "Article DOI",
    "PMID",
    "Patent Number",
    "BindingDB Entry DOI",
    "PubChem AID",
]


def normalize_doi(x):
    x = clean(x).lower()

    if not x:
        return ""

    x = re.sub(
        r"^https?://(?:dx\.)?doi\.org/",
        "",
        x
    )

    x = re.sub(
        r"^doi:\s*",
        "",
        x
    )

    return x.strip()


def normalize_pmid(x):
    x = clean(x)

    if not x:
        return ""

    # remove accidental .0
    x = re.sub(r"\.0$", "", x)

    return x


def normalize_patent(x):
    x = clean(x).upper()

    if not x:
        return ""

    # remove spaces for more consistent comparison
    x = re.sub(r"\s+", "", x)

    return x


def normalize_aid(x):
    x = clean(x)

    if not x:
        return ""

    x = re.sub(r"\.0$", "", x)

    return x


def make_source_token(column, value):

    if column == "Article DOI":
        v = normalize_doi(value)
        return f"ARTICLE_DOI:{v}" if v else None

    if column == "PMID":
        v = normalize_pmid(value)
        return f"PMID:{v}" if v else None

    if column == "Patent Number":
        v = normalize_patent(value)
        return f"PATENT:{v}" if v else None

    if column == "BindingDB Entry DOI":
        v = normalize_doi(value)
        return f"BDB_ENTRY_DOI:{v}" if v else None

    if column == "PubChem AID":
        v = normalize_aid(value)
        return f"PUBCHEM_AID:{v}" if v else None

    return None


# ============================================================
# UNION-FIND
#
# This allows:
#
# Row 1:
# DOI = X
# PMID = 123
#
# Row 2:
# DOI = blank
# PMID = 123
#
# to still be recognized as SAME SOURCE.
# ============================================================

class UnionFind:

    def __init__(self):
        self.parent = {}
        self.rank = {}

    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )
        return self.parent[x]

    def union(self, a, b):

        self.add(a)
        self.add(b)

        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra

        self.parent[rb] = ra

        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


# ============================================================
# TARGET PARSING
# ============================================================

# One-letter substitutions:
# T790M
# L858R
# V600E
# T315I
SUB_PATTERN = re.compile(
    r"(?<![A-Za-z0-9])"
    r"([ACDEFGHIKLMNPQRSTVWY])"
    r"(\d+)"
    r"([ACDEFGHIKLMNPQRSTVWY*])"
    r"(?![A-Za-z0-9])",
    re.I
)

# Common deletion/insertion formats
DEL_INS_PATTERN = re.compile(
    r"""
    (?:
        [A-Z]?\d+
        (?:_[A-Z]?\d+)?
        (?:delins|del|ins|dup)
        [A-Z]*
    )
    """,
    re.I | re.X
)

# Explicit mutation-related language
MUTANT_WORD_PATTERN = re.compile(
    r"""
    \bmutant\b
    |\bmutation\b
    |\bmutated\b
    |\bdeletion\b
    |\binsertion\b
    |\bsubstitution\b
    |Δ
    """,
    re.I | re.X
)

WT_PATTERN = re.compile(
    r"""
    \bwild[\s-]*type\b
    |\bWT\b
    """,
    re.I | re.X
)


def get_annotation(target):

    target = clean(target)

    annotations = re.findall(
        r"\[([^\]]+)\]",
        target
    )

    if annotations:
        return " | ".join(annotations)

    return ""


def get_base_target(target):

    target = clean(target)

    if not target:
        return ""

    # Remove BindingDB-style [construct/mutation] annotation
    base = re.sub(
        r"\s*\[[^\]]*\]",
        "",
        target
    )

    # Remove obvious trailing WT labels
    base = re.sub(
        r"\s*\((?:WT|wild[\s-]*type)\)\s*$",
        "",
        base,
        flags=re.I
    )

    base = re.sub(
        r"\s+",
        " ",
        base
    ).strip()

    return base


def parse_variant(target):

    target = clean(target)

    if not target:
        return "Unknown", ""

    mutations = []

    # substitutions
    for m in SUB_PATTERN.finditer(target):
        mutations.append(
            f"{m.group(1).upper()}"
            f"{m.group(2)}"
            f"{m.group(3).upper()}"
        )

    # deletions / insertions
    for m in DEL_INS_PATTERN.finditer(target):
        mutations.append(m.group(0))

    # remove duplicates
    mutations = list(dict.fromkeys(mutations))

    if mutations:
        return "MT", ";".join(mutations)

    # target says mutant, but exact mutation couldn't be parsed
    if MUTANT_WORD_PATTERN.search(target):
        return "MT_unparsed", ""

    # explicitly says WT
    if WT_PATTERN.search(target):
        return "WT_explicit", ""

    # no mutation annotation
    # Important: this is NOT proof of WT.
    return "WT_candidate", ""


# ============================================================
# LIGAND ID
#
# Prefer BindingDB MonomerID.
# Fallback to InChIKey, SMILES, ligand name.
# ============================================================

def make_ligand_id(row):

    monomer = clean(
        row.get("BindingDB MonomerID", "")
    )

    if monomer:
        monomer = re.sub(r"\.0$", "", monomer)
        return "BDBM:" + monomer

    for possible_col in [
        "Ligand InChI key",
        "Ligand InChI Key"
    ]:
        inchikey = clean(row.get(possible_col, ""))

        if inchikey:
            return "INCHIKEY:" + inchikey.upper()

    smiles = clean(
        row.get("Ligand SMILES", "")
    )

    if smiles:
        return "SMILES:" + smiles

    name = clean(
        row.get("BindingDB Ligand Name", "")
    )

    if name:
        return "NAME:" + name.lower()

    return ""


# ============================================================
# GET SOURCE TOKENS FROM ROW
# ============================================================

def get_source_tokens(row, available_source_cols):

    tokens = []

    for col in available_source_cols:

        token = make_source_token(
            col,
            row.get(col, "")
        )

        if token:
            tokens.append(token)

    return list(dict.fromkeys(tokens))


# ============================================================
# PROCESS ONE FILE
# ============================================================

def process_file(label, input_file, output_file):

    print("\n")
    print("=" * 80)
    print(label)
    print("=" * 80)

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    header = pd.read_csv(
        input_file,
        compression="gzip",
        nrows=0
    ).columns.tolist()

    if "Target Name" not in header:
        raise ValueError(
            f"'Target Name' missing from {input_file}"
        )

    if "Kd (nM)" not in header:
        raise ValueError(
            f"'Kd (nM)' missing from {input_file}"
        )

    available_source_cols = [
        c for c in SOURCE_COLUMNS
        if c in header
    ]

    print("Source identifiers available:")

    for c in available_source_cols:
        print("  ", c)

    if not available_source_cols:
        raise ValueError(
            "No source identifier columns found."
        )


    # ========================================================
    # PASS 1
    #
    # Link all source identifiers together.
    # ========================================================

    uf = UnionFind()

    print("\nPASS 1: building source relationships...")

    for chunk_num, chunk in enumerate(
        pd.read_csv(
            input_file,
            compression="gzip",
            usecols=available_source_cols,
            dtype=str,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        1
    ):

        for row in chunk.to_dict("records"):

            tokens = get_source_tokens(
                row,
                available_source_cols
            )

            if not tokens:
                continue

            for token in tokens:
                uf.add(token)

            first = tokens[0]

            for token in tokens[1:]:
                uf.union(first, token)

        print(
            f"  source pass chunk {chunk_num}"
        )


    # --------------------------------------------------------
    # Convert connected source IDs to simple SRC IDs
    # --------------------------------------------------------

    roots = {}

    for token in uf.parent:
        root = uf.find(token)

        if root not in roots:
            roots[root] = None

    for i, root in enumerate(
        sorted(roots),
        start=1
    ):
        roots[root] = f"SRC_{i:07d}"

    token_to_source_group = {
        token: roots[uf.find(token)]
        for token in uf.parent
    }

    print(
        "Unique source groups:",
        len(roots)
    )


    # ========================================================
    # Helper to assign Source_Group_ID
    # ========================================================

    def get_source_group(row):

        tokens = get_source_tokens(
            row,
            available_source_cols
        )

        for token in tokens:

            if token in token_to_source_group:
                return token_to_source_group[token]

        return ""


    # ========================================================
    # PASS 2
    #
    # Build a SMALL index:
    #
    # source
    # ligand
    # target
    # WT/MT
    #
    # We do NOT load all BindingDB fields here.
    # ========================================================

    print("\nPASS 2: finding WT/MT comparison groups...")

    slim_cols = list(
        dict.fromkeys(
            available_source_cols
            + [
                "BindingDB MonomerID",
                "Ligand InChI key",
                "Ligand InChI Key",
                "Ligand SMILES",
                "BindingDB Ligand Name",
                "Target Name",
                "Kd (nM)"
            ]
        )
    )

    slim_cols = [
        c for c in slim_cols
        if c in header
    ]

    summaries = []

    for chunk_num, chunk in enumerate(
        pd.read_csv(
            input_file,
            compression="gzip",
            usecols=slim_cols,
            dtype=str,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        1
    ):

        # Verify Kd
        kd = (
            chunk["Kd (nM)"]
            .astype("string")
            .str.strip()
        )

        chunk = chunk[
            kd.notna()
            & (kd != "")
            & (kd.str.lower() != "nan")
        ].copy()

        if chunk.empty:
            continue


        records = []

        for row in chunk.to_dict("records"):

            source_group = get_source_group(row)

            # Cannot verify "same source"
            if not source_group:
                continue

            ligand_id = make_ligand_id(row)

            if not ligand_id:
                continue

            target_name = clean(
                row.get("Target Name", "")
            )

            base_target = get_base_target(
                target_name
            )

            if not base_target:
                continue

            variant_status, mutation = (
                parse_variant(target_name)
            )

            records.append({
                "Source_Group_ID":
                    source_group,

                "Ligand_Group_ID":
                    ligand_id,

                "Target_Base":
                    base_target,

                "Variant_Status":
                    variant_status
            })


        if not records:
            continue

        temp = pd.DataFrame(records)

        temp["is_wt"] = (
            temp["Variant_Status"]
            .isin([
                "WT_candidate",
                "WT_explicit"
            ])
        ).astype(int)

        temp["is_mt"] = (
            temp["Variant_Status"]
            .isin([
                "MT",
                "MT_unparsed"
            ])
        ).astype(int)


        g = (
            temp.groupby(
                [
                    "Source_Group_ID",
                    "Ligand_Group_ID",
                    "Target_Base"
                ],
                dropna=False
            )
            .agg(
                N_rows=("Variant_Status", "size"),
                N_WT=("is_wt", "sum"),
                N_MT=("is_mt", "sum")
            )
            .reset_index()
        )

        summaries.append(g)

        print(
            f"  group pass chunk {chunk_num}"
        )


    # --------------------------------------------------------
    # Combine chunk summaries
    # --------------------------------------------------------

    summary = pd.concat(
        summaries,
        ignore_index=True
    )

    summary = (
        summary.groupby(
            [
                "Source_Group_ID",
                "Ligand_Group_ID",
                "Target_Base"
            ],
            as_index=False
        )
        [["N_rows", "N_WT", "N_MT"]]
        .sum()
    )


    # ========================================================
    # CRITICAL FILTER
    #
    # Must contain BOTH:
    #   >=1 WT
    #   >=1 MT
    #
    # Therefore:
    # WT-only groups disappear.
    # MT-only groups disappear.
    # ========================================================

    matched = summary[
        (summary["N_WT"] >= 1)
        &
        (summary["N_MT"] >= 1)
    ].copy()


    print(
        "\nComparison groups containing WT + MT:",
        len(matched)
    )

    print(
        "WT rows represented:",
        matched["N_WT"].sum()
    )

    print(
        "MT rows represented:",
        matched["N_MT"].sum()
    )


    # Tuple set for fast filtering
    matched_keys = set(
        zip(
            matched["Source_Group_ID"],
            matched["Ligand_Group_ID"],
            matched["Target_Base"]
        )
    )


    # stats lookup
    stats_lookup = {
        (
            r.Source_Group_ID,
            r.Ligand_Group_ID,
            r.Target_Base
        ): (
            r.N_rows,
            r.N_WT,
            r.N_MT
        )

        for r in matched.itertuples(
            index=False
        )
    }


    # ========================================================
    # PASS 3
    #
    # Read FULL original rows and write only matched groups.
    # ========================================================

    print("\nPASS 3: writing full matched BindingDB rows...")

    first = True
    n_written = 0


    for chunk_num, chunk in enumerate(
        pd.read_csv(
            input_file,
            compression="gzip",
            dtype=str,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        1
    ):

        output_rows = []

        for row in chunk.to_dict("records"):

            # ensure Kd exists
            kd = clean(
                row.get("Kd (nM)", "")
            )

            if not kd:
                continue

            source_group = get_source_group(row)

            if not source_group:
                continue

            ligand_id = make_ligand_id(row)

            if not ligand_id:
                continue

            target_name = clean(
                row.get("Target Name", "")
            )

            target_base = get_base_target(
                target_name
            )

            key = (
                source_group,
                ligand_id,
                target_base
            )

            if key not in matched_keys:
                continue

            variant_status, mutation = (
                parse_variant(target_name)
            )

            # Should only be WT or MT records
            if variant_status not in [
                "WT_candidate",
                "WT_explicit",
                "MT",
                "MT_unparsed"
            ]:
                continue

            n_rows, n_wt, n_mt = (
                stats_lookup[key]
            )

            # Preserve EVERY original column
            # and add our derived fields
            row["Source_Group_ID"] = (
                source_group
            )

            row["Ligand_Group_ID"] = (
                ligand_id
            )

            row["Target_Base"] = (
                target_base
            )

            row["Target_Annotation"] = (
                get_annotation(target_name)
            )

            row["Variant_Status"] = (
                variant_status
            )

            row["Mutation"] = (
                mutation
            )

            row["WT_MT_Group_N"] = (
                n_rows
            )

            row["N_WT_in_Group"] = (
                n_wt
            )

            row["N_MT_in_Group"] = (
                n_mt
            )

            output_rows.append(row)


        if output_rows:

            out = pd.DataFrame(
                output_rows
            )

            out.to_csv(
                output_file,
                index=False,
                compression="gzip",
                mode="wt" if first else "at",
                header=first
            )

            first = False
            n_written += len(out)


        print(
            f"  write chunk {chunk_num} | "
            f"rows written={n_written:,}"
        )


    print("\nFINISHED:", label)
    print("Rows written:", f"{n_written:,}")
    print("File:", output_file)

    if os.path.exists(output_file):

        print(
            "Compressed size:",
            f"{os.path.getsize(output_file)/1024**2:.1f} MB"
        )

    return matched


# ============================================================
# RUN BOTH
# ============================================================

results = {}

for label, paths in FILES.items():

    results[label] = process_file(
        label,
        paths["input"],
        paths["output"]
    )


print("\n")
print("=" * 80)
print("ALL DONE")
print("=" * 80)

for label, paths in FILES.items():
    print(
        label,
        "->",
        paths["output"]
    )



Kinase
Source identifiers available:
   Article DOI
   PMID
   Patent Number
   BindingDB Entry DOI
   PubChem AID

PASS 1: building source relationships...
  source pass chunk 1
Unique source groups: 792

PASS 2: finding WT/MT comparison groups...
  group pass chunk 1

Comparison groups containing WT + MT: 500
WT rows represented: 622
MT rows represented: 763

PASS 3: writing full matched BindingDB rows...
  write chunk 1 | rows written=1,385

FINISHED: Kinase
Rows written: 1,385
File: /content/BindingDB_Kinase_Kd_WT_MT_same_source.csv.gz
Compressed size: 0.2 MB


Metabolic enzyme
Source identifiers available:
   Article DOI
   PMID
   Patent Number
   BindingDB Entry DOI
   PubChem AID

PASS 1: building source relationships...
  source pass chunk 1
Unique source groups: 816

PASS 2: finding WT/MT comparison groups...
  group pass chunk 1

Comparison groups containing WT + MT: 27
WT rows represented: 27
MT rows represented: 45

PASS 3: writing full matched BindingDB rows...
  write 

In [ ]:
import pandas as pd
import gzip

files = {
    "Kinase": "/content/BindingDB_Kinase_Kd_WT_MT_same_source.csv.gz",
    "Metabolic enzyme": "/content/BindingDB_MetabolicEnzyme_Kd_WT_MT_same_source.csv.gz",
}

for name, path in files.items():

    print("\n" + "="*100)
    print(name)
    print("="*100)

    # Count rows without loading everything
    with gzip.open(path, "rt") as f:
        n_rows = sum(1 for _ in f) - 1   # subtract header

    print(f"Total rows: {n_rows:,}")

    # Read first 10 rows
    df10 = pd.read_csv(
        path,
        compression="gzip",
        nrows=10,
        low_memory=False
    )

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 80)

    display(df10)


Kinase
Total rows: 1,385


,BindingDB Reactant_set_id,Ligand SMILES,Ligand InChI,Ligand InChI Key,BindingDB MonomerID,BindingDB Ligand Name,Target Name,Target Source Organism According to Curator or DataSource,Ki (nM),IC50 (nM),Kd (nM),EC50 (nM),kon (M-1-s-1),koff (s-1),pH,Temp (C),Curation/DataSource,Article DOI,BindingDB Entry DOI,PMID,PubChem AID,Patent Number,Authors,Date of publication,Date in BindingDB,Institution,Link to Ligand in BindingDB,Link to Target in BindingDB,Link to Ligand-Target Pair in BindingDB,Ligand HET ID in PDB,PDB ID(s) for Ligand-Target Complex,PubChem CID,PubChem SID,ChEBI ID of Ligand,ChEMBL ID of Ligand,DrugBank ID of Ligand,IUPHAR_GRAC ID of Ligand,KEGG ID of Ligand,ZINC ID of Ligand,Number of Protein Chains in Target (>1 implies a multichain complex),BindingDB Target Chain Sequence 1,PDB ID(s) of Target Chain 1,UniProt (SwissProt) Recommended Name of Target Chain 1,UniProt (SwissProt) Entry Name of Target Chain 1,UniProt (SwissProt) Primary ID of Target Chain 1,UniProt (SwissProt) Secondary ID(s) of Target Chain 1,UniProt (SwissProt) Alternative ID(s) of Target Chain 1,UniProt (TrEMBL) Submitted Name of Target Chain 1,UniProt (TrEMBL) Entry Name of Target Chain 1,UniProt (TrEMBL) Primary ID of Target Chain 1,UniProt (TrEMBL) Secondary ID(s) of Target Chain 1,UniProt (TrEMBL) Alternative ID(s) of Target Chain 1,BindingDB Target Chain Sequence 2,PDB ID(s) of Target Chain 2,UniProt (SwissProt) Recommended Name of Target Chain 2,UniProt (SwissProt) Entry Name of Target Chain 2,UniProt (SwissProt) Primary ID of Target Chain 2,UniProt (SwissProt) Secondary ID(s) of Target Chain 2,UniProt (SwissProt) Alternative ID(s) of Target Chain 2,UniProt (TrEMBL) Submitted Name of Target Chain 2,UniProt (TrEMBL) Entry Name of Target Chain 2,UniProt (TrEMBL) Primary ID of Target Chain 2,UniProt (TrEMBL) Secondary ID(s) of Target Chain 2,UniProt (TrEMBL) Alternative ID(s) of Target Chain 2,BindingDB Target Chain Sequence 3,PDB ID(s) of Target Chain 3,UniProt (SwissProt) Recommended Name of Target Chain 3,UniProt (SwissProt) Entry Name of Target Chain 3,UniProt (SwissProt) Primary ID of Target Chain 3,UniProt (SwissProt) Secondary ID(s) of Target Chain 3,UniProt (SwissProt) Alternative ID(s) of Target Chain 3,UniProt (TrEMBL) Submitted Name of Target Chain 3,UniProt (TrEMBL) Entry Name of Target Chain 3,UniProt (TrEMBL) Primary ID of Target Chain 3,UniProt (TrEMBL) Secondary ID(s) of Target Chain 3,UniProt (TrEMBL) Alternative ID(s) of Target Chain 3,BindingDB Target Chain Sequence 4,PDB ID(s) of Target Chain 4,UniProt (SwissProt) Recommended Name of Target Chain 4,UniProt (SwissProt) Entry Name of Target Chain 4,UniProt (SwissProt) Primary ID of Target Chain 4,UniProt (SwissProt) Secondary ID(s) of Target Chain 4,UniProt (SwissProt) Alternative ID(s) of Target Chain 4,UniProt (TrEMBL) Submitted Name of Target Chain 4,UniProt (TrEMBL) Entry Name of Target Chain 4,UniProt (TrEMBL) Primary ID of Target Chain 4,UniProt (TrEMBL) Secondary ID(s) of Target Chain 4,UniProt (TrEMBL) Alternative ID(s) of Target Chain 4,BindingDB Target Chain Sequence 5,PDB ID(s) of Target Chain 5,UniProt (SwissProt) Recommended Name of Target Chain 5,UniProt (SwissProt) Entry Name of Target Chain 5,UniProt (SwissProt) Primary ID of Target Chain 5,UniProt (SwissProt) Secondary ID(s) of Target Chain 5,UniProt (SwissProt) Alternative ID(s) of Target Chain 5,UniProt (TrEMBL) Submitted Name of Target Chain 5,UniProt (TrEMBL) Entry Name of Target Chain 5,UniProt (TrEMBL) Primary ID of Target Chain 5,UniProt (TrEMBL) Secondary ID(s) of Target Chain 5,UniProt (TrEMBL) Alternative ID(s) of Target Chain 5,BindingDB Target Chain Sequence 6,PDB ID(s) of Target Chain 6,UniProt (SwissProt) Recommended Name of Target Chain 6,UniProt (SwissProt) Entry Name of Target Chain 6,UniProt (SwissProt) Primary ID of Target Chain 6,UniProt (SwissProt) Secondary ID(s) of Target Chain 6,UniProt (SwissProt) Alternative ID(s) of Target Chain 6,UniProt (TrEMBL) Submitted Name of Target Chain 6,UniPr


Metabolic enzyme
Total rows: 72


,BindingDB Reactant_set_id,Ligand SMILES,Ligand InChI,Ligand InChI Key,BindingDB MonomerID,BindingDB Ligand Name,Target Name,Target Source Organism According to Curator or DataSource,Ki (nM),IC50 (nM),Kd (nM),EC50 (nM),kon (M-1-s-1),koff (s-1),pH,Temp (C),Curation/DataSource,Article DOI,BindingDB Entry DOI,PMID,PubChem AID,Patent Number,Authors,Date of publication,Date in BindingDB,Institution,Link to Ligand in BindingDB,Link to Target in BindingDB,Link to Ligand-Target Pair in BindingDB,Ligand HET ID in PDB,PDB ID(s) for Ligand-Target Complex,PubChem CID,PubChem SID,ChEBI ID of Ligand,ChEMBL ID of Ligand,DrugBank ID of Ligand,IUPHAR_GRAC ID of Ligand,KEGG ID of Ligand,ZINC ID of Ligand,Number of Protein Chains in Target (>1 implies a multichain complex),BindingDB Target Chain Sequence 1,PDB ID(s) of Target Chain 1,UniProt (SwissProt) Recommended Name of Target Chain 1,UniProt (SwissProt) Entry Name of Target Chain 1,UniProt (SwissProt) Primary ID of Target Chain 1,UniProt (SwissProt) Secondary ID(s) of Target Chain 1,UniProt (SwissProt) Alternative ID(s) of Target Chain 1,UniProt (TrEMBL) Submitted Name of Target Chain 1,UniProt (TrEMBL) Entry Name of Target Chain 1,UniProt (TrEMBL) Primary ID of Target Chain 1,UniProt (TrEMBL) Secondary ID(s) of Target Chain 1,UniProt (TrEMBL) Alternative ID(s) of Target Chain 1,BindingDB Target Chain Sequence 2,PDB ID(s) of Target Chain 2,UniProt (SwissProt) Recommended Name of Target Chain 2,UniProt (SwissProt) Entry Name of Target Chain 2,UniProt (SwissProt) Primary ID of Target Chain 2,UniProt (SwissProt) Secondary ID(s) of Target Chain 2,UniProt (SwissProt) Alternative ID(s) of Target Chain 2,UniProt (TrEMBL) Submitted Name of Target Chain 2,UniProt (TrEMBL) Entry Name of Target Chain 2,UniProt (TrEMBL) Primary ID of Target Chain 2,UniProt (TrEMBL) Secondary ID(s) of Target Chain 2,UniProt (TrEMBL) Alternative ID(s) of Target Chain 2,BindingDB Target Chain Sequence 3,PDB ID(s) of Target Chain 3,UniProt (SwissProt) Recommended Name of Target Chain 3,UniProt (SwissProt) Entry Name of Target Chain 3,UniProt (SwissProt) Primary ID of Target Chain 3,UniProt (SwissProt) Secondary ID(s) of Target Chain 3,UniProt (SwissProt) Alternative ID(s) of Target Chain 3,UniProt (TrEMBL) Submitted Name of Target Chain 3,UniProt (TrEMBL) Entry Name of Target Chain 3,UniProt (TrEMBL) Primary ID of Target Chain 3,UniProt (TrEMBL) Secondary ID(s) of Target Chain 3,UniProt (TrEMBL) Alternative ID(s) of Target Chain 3,BindingDB Target Chain Sequence 4,PDB ID(s) of Target Chain 4,UniProt (SwissProt) Recommended Name of Target Chain 4,UniProt (SwissProt) Entry Name of Target Chain 4,UniProt (SwissProt) Primary ID of Target Chain 4,UniProt (SwissProt) Secondary ID(s) of Target Chain 4,UniProt (SwissProt) Alternative ID(s) of Target Chain 4,UniProt (TrEMBL) Submitted Name of Target Chain 4,UniProt (TrEMBL) Entry Name of Target Chain 4,UniProt (TrEMBL) Primary ID of Target Chain 4,UniProt (TrEMBL) Secondary ID(s) of Target Chain 4,UniProt (TrEMBL) Alternative ID(s) of Target Chain 4,BindingDB Target Chain Sequence 5,PDB ID(s) of Target Chain 5,UniProt (SwissProt) Recommended Name of Target Chain 5,UniProt (SwissProt) Entry Name of Target Chain 5,UniProt (SwissProt) Primary ID of Target Chain 5,UniProt (SwissProt) Secondary ID(s) of Target Chain 5,UniProt (SwissProt) Alternative ID(s) of Target Chain 5,UniProt (TrEMBL) Submitted Name of Target Chain 5,UniProt (TrEMBL) Entry Name of Target Chain 5,UniProt (TrEMBL) Primary ID of Target Chain 5,UniProt (TrEMBL) Secondary ID(s) of Target Chain 5,UniProt (TrEMBL) Alternative ID(s) of Target Chain 5,BindingDB Target Chain Sequence 6,PDB ID(s) of Target Chain 6,UniProt (SwissProt) Recommended Name of Target Chain 6,UniProt (SwissProt) Entry Name of Target Chain 6,UniProt (SwissProt) Primary ID of Target Chain 6,UniProt (SwissProt) Secondary ID(s) of Target Chain 6,UniProt (SwissProt) Alternative ID(s) of Target Chain 6,UniProt (TrEMBL) Submitted Name of Target Chain 6,UniPr

In [ ]:
for name, path in files.items():

    counts = {}

    for chunk in pd.read_csv(
        path,
        compression="gzip",
        usecols=["Variant_Status"],
        chunksize=100_000
    ):
        vc = chunk["Variant_Status"].value_counts()

        for status, n in vc.items():
            counts[status] = counts.get(status, 0) + n

    print(f"\n{name}")
    for status, n in counts.items():
        print(f"  {status}: {n:,}")


Kinase
  MT: 763
  WT_candidate: 622

Metabolic enzyme
  MT: 45
  WT_candidate: 27


In [ ]:
for name, path in files.items():

    groups = set()

    for chunk in pd.read_csv(
        path,
        compression="gzip",
        usecols=[
            "Source_Group_ID",
            "Ligand_Group_ID",
            "Target_Base"
        ],
        chunksize=100_000
    ):

        groups.update(
            zip(
                chunk["Source_Group_ID"],
                chunk["Ligand_Group_ID"],
                chunk["Target_Base"]
            )
        )

    print(f"{name}: {len(groups):,} WT/MT comparison groups")

Kinase: 500 WT/MT comparison groups
Metabolic enzyme: 27 WT/MT comparison groups
